## Character-Level Text Generation

In [1]:
import tensorflow as tf
import numpy as np
import os
import time

### 1. Download the dataset

In [2]:
path_to_file = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

text = open(path_to_file, 'rb').read().decode(encoding='utf-8')

print(f'Length of text: {len(text)} characters')
print(text[:250])

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Length of text: 1115394 characters
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



###  Process the text

In [3]:
# unique characters in the text
vocab = sorted(list(set(text)))
print(f'{len(vocab)} unique characters')

# Create mappings from character to integer and integer to character
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = np.array(vocab)

text_as_int = np.array([char_to_idx[char] for char in text])

print('{')
for char, _ in zip(char_to_idx, range(20)):
    print(f'  {repr(char)}: {char_to_idx[char]},')
print('...\n}')

print(f'{repr(text[:13])} ---- characters mapped to int ----> {text_as_int[:13]}')

65 unique characters
{
  '\n': 0,
  ' ': 1,
  '!': 2,
  '$': 3,
  '&': 4,
  "'": 5,
  ',': 6,
  '-': 7,
  '.': 8,
  '3': 9,
  ':': 10,
  ';': 11,
  '?': 12,
  'A': 13,
  'B': 14,
  'C': 15,
  'D': 16,
  'E': 17,
  'F': 18,
  'G': 19,
...
}
'First Citizen' ---- characters mapped to int ----> [18 47 56 57 58  1 15 47 58 47 64 43 52]


In [4]:
# The maximum length sentence we want for a single input in characters
seq_length = 100
examples_per_epoch = len(text_as_int) // (seq_length + 1)

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

for i in char_dataset.take(5):
    print(idx_to_char[i.numpy()])

sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

for item in sequences.take(5):
    print(repr(''.join(idx_to_char[item.numpy()])))

F
i
r
s
t
'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '
'are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you k'
"now Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us ki"
"ll him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be d"
'one: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citi'


In [5]:
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

for input_example, target_example in dataset.take(1):
    print('Input : ', repr(''.join(idx_to_char[input_example.numpy()])))
    print('Target: ', repr(''.join(idx_to_char[target_example.numpy()])))

BATCH_SIZE = 64

BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE))

dataset

Input :  'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'
Target:  'irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '


<_PrefetchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>

### Build model

In [6]:
vocab_size = len(vocab)

embedding_dim = 256

# Number of RNN units
rnn_units = 512

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(batch_size=batch_size, shape=(None,)),
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.GRU(rnn_units,
                            return_sequences=True,
                            stateful=True,
                            recurrent_initializer='glorot_uniform'),

        tf.keras.layers.Dense(vocab_size)
    ])
    return model

model = build_model(
    vocab_size=len(vocab),
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
    batch_size=BATCH_SIZE)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (64, None, 256)        │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (64, None, 512)        │     1,182,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, None, 65)         │        33,345 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,232,705 (4.70 MB)

 Trainable params: 1,232,705 (4.70 MB)

 Non-trainable params: 0 (0.00 B)

### Train the model

In [7]:
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")

# Evaluate the model on the example batch
sampled_indices = tf.random.categorical(example_batch_predictions[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices, axis=-1).numpy()
print(f"Predictions for one sample: {sampled_indices}")
print(f"Characters generated: {''.join(idx_to_char[sampled_indices])}")

def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels,
                                                           logits,
                                                           from_logits=True)

example_batch_loss = loss(target_example_batch, example_batch_predictions)
print(f"Loss for one sample: {example_batch_loss.numpy().mean()}")

model.compile(optimizer='adam', loss=loss)

# Directory where the checkpoints will be saved
checkpoint_dir = './training_checkpoints'
# Name of the checkpoint files
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_weights_only=True)

EPOCHS = 20
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

(64, 100, 65) # (batch_size, sequence_length, vocab_size)
Predictions for one sample: [60 26  2 41  2 46  8 28 41 13 45 39 12 13 45 33 54 30  2 19  7 41 46  9
 33 17 48 17 48 22 23 11 30 44  5 35  3 28 22 14 46 36 12 61  4 53 24 21
 41 13 60 20 47 52 30 37 28 55 37 28 44 16 33 47  0 47 49 57  0 61 17 40
 49 25 11 52  2 55 55  1 30 32 35 38 46  2 54 60 11 12 52 57 16 51 20 10
 45 19  5  5]
Characters generated: vN!c!h.PcAga?AgUpR!G-ch3UEjEjJK;Rf'W$PJBhX?w&oLIcAvHinRYPqYPfDUi
iks
wEbkM;n!qq RTWZh!pv;?nsDmH:gG''
Loss for one sample: 4.173949241638184
Epoch 1/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - loss: 2.4169
Epoch 2/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - loss: 1.8480
Epoch 3/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - loss: 1.6399
Epoch 4/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - loss: 1.5323
Epoch 5/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - loss: 1.4668
Epoch 6/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - loss: 1.4223
Epoch 7/20
172/172 ━━━━━━━━━━━━━━━━━━

### 6. Generate text

In [9]:
import glob

latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)

if latest_checkpoint is None:
    list_of_files = glob.glob(os.path.join(checkpoint_dir, 'ckpt_*.weights.h5'))
    if list_of_files:
        latest_checkpoint = max(list_of_files, key=os.path.getctime)
    else:
        raise FileNotFoundError(f"No checkpoint files found in {checkpoint_dir}")

model = build_model(vocab_size, embedding_dim, rnn_units, batch_size=1)

model.load_weights(latest_checkpoint)

model.build(tf.TensorShape([1, None]))

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (1, None, 256)         │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (1, None, 512)         │     1,182,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (1, None, 65)          │        33,345 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,232,705 (4.70 MB)

 Trainable params: 1,232,705 (4.70 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
def generate_text(model, start_string):
    # Number of char to generate
    num_generate = 1000

    # Convert our start string to numbers (vectorize)
    input_eval = [char_to_idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    # Low temperatures -  more predictable text
    # Higher temperatures - more random text
    temperature = 1.0

    # Here batch size = 1, as text will be generated one char at time
    model.layers[1].reset_states()
    for i in range(num_generate):
        predictions = model(input_eval)
        # Remove the batch dimension
        predictions = tf.squeeze(predictions, 0)

        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()

        # ppass the predicted character as the next input to the model along with the previous hidden state
        input_eval = tf.expand_dims([predicted_id], 0)

        text_generated.append(idx_to_char[predicted_id])

    return start_string + ''.join(text_generated)

In [19]:
print(generate_text(model, start_string="ROMEO:"))

ROMEO:
Go, get you yet, not for the mids, how fares it me, in thus to our purposes
Cannow where he is of all my will not fear it, great friends,
And show your couraties and you takes him to;
To joy conscience shall be Barkn, lords an one that kiss the affections
But Romeo comes?

FRIAR LAURENCE:
I have inon a grain of fie infuries who
slaughter'd her,
A right repointed creature tooous shart with
this deed answers with him.

First Lady:
Go, room thy spoit ar his own
royalty shall be company;
And I will say we'll hear that title had so with
the match to find his paduatimn and yourselves;
And prated in many fool, and then a paw of Rome are dead!
The king's vartity, or replied, but farewell.

HASTINGS:
Away with her; oy, so not to kiss again;
And I with good assistance. How will stay away.
3 KING HENRY VI:
Be patient; for they thy beadst in this.

First Lord:
From she's such leave on this ce horse! O heavy love!

PAULINA:
And to confuse
A fearful tears, and all the hour, his mind
on peace?